# 1002 · 02 · Silver: clean and validate

Silver turns bronze into data you can **trust**:

1. Give every column its real **type** (numbers, timestamps).
2. Apply the **quality rules**, one by one, and count what each rule removes.
3. Write only the rows that pass.

The two small sources (turbines and work orders) only need types. The SCADA
readings are where the problems are.

In [ ]:
%python
from pyspark.sql import functions as F

CATALOG = "lab1002"

## 1 · Master data and work orders: just types

In [ ]:
%python
turbines = (spark.table(f"{CATALOG}.bronze.turbines")
            .select("turbine_id", "park_id", "park", "model",
                    F.col("rated_power_mw").cast("double").alias("rated_power_mw")))

work_orders = (spark.table(f"{CATALOG}.bronze.work_orders")
               .select("work_order_id", "turbine_id", "reason",
                       F.to_timestamp("start_time").alias("start_time"),
                       F.to_timestamp("end_time").alias("end_time")))

turbines.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.turbines")
work_orders.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.work_orders")
print(turbines.count(), "turbines |", work_orders.count(), "work orders")

## 2 · SCADA readings: types

In [ ]:
%python
scada = (spark.table(f"{CATALOG}.bronze.scada")
         .select(F.to_timestamp("reading_time").alias("reading_time"),
                 "turbine_id",
                 F.col("wind_speed_ms").cast("double").alias("wind_speed_ms"),
                 F.col("power_kw").cast("double").alias("power_kw"),
                 "status", "_ingested_at", "_job_run_id"))

rows_in = scada.count()
print(rows_in, "readings from bronze")

## 3 · The quality rules

Each rule is one line, applied to the output of the previous one:

| # | Rule | Why |
|---|---|---|
| 1 | One reading per turbine and time | The SCADA export sometimes sends a row twice |
| 2 | Wind and power are not empty | A reading without values is useless |
| 3 | Wind between 0 and 40 m/s | Above 40 m/s is a broken anemometer, not weather |
| 4 | The turbine exists in the master data | Readings from an unknown turbine cannot be assigned to a wind farm |
| 5 | Power between 0 and 105 % of the turbine's rated power | A turbine cannot produce more than it was built for |

In [ ]:
%python
r1 = scada.dropDuplicates(["turbine_id", "reading_time"])
r2 = r1.filter(F.col("wind_speed_ms").isNotNull() & F.col("power_kw").isNotNull())
r3 = r2.filter(F.col("wind_speed_ms").between(0, 40))
r4 = r3.join(turbines.select("turbine_id", "rated_power_mw"), "turbine_id")
r5 = r4.filter(F.col("power_kw").between(0, F.col("rated_power_mw") * 1000 * 1.05))

clean = r5.drop("rated_power_mw")

## 4 · What each rule removed

In [ ]:
%python
counts = [rows_in] + [df.count() for df in (r1, r2, r3, r4, r5)]
rules = ["1 · duplicates", "2 · empty values", "3 · impossible wind",
         "4 · unknown turbine", "5 · impossible power"]

report = spark.createDataFrame(
    [(rule, counts[i] - counts[i + 1]) for i, rule in enumerate(rules)]
    + [("rows kept", counts[-1])],
    ["rule", "rows"])
display(report)

## 5 · Write the silver table

In [ ]:
%python
clean.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.scada")
print(spark.table(f"{CATALOG}.silver.scada").count(), "readings in silver.scada")